# Notebook 2 — Baseline Demand Forecasting (GBTRegressor)
**SmartRetail: Predictive Demand Forecasting & Dynamic Pricing Engine**

### What this notebook does
1. Loads the `train/` and `test/` Parquet datasets written by notebook 1
2. Engineers lag features (`views_lag1`, `addtocart_lag1`) so the model never sees same-day intent signals
3. Drops columns that would leak same-day information (`daily_views`, `daily_addtocarts`, `view_to_cart_ratio`, `cart_to_purchase_ratio`)
4. Restricts training (and evaluation) to SKUs that have at least one historical sale, avoiding a model that trivially predicts zero
5. Encodes `itemid` and `categoryid` with `StringIndexer` and strips the resulting nominal metadata so the tree model treats them as ordinary numeric splits (avoids the `maxBins` ceiling on high-cardinality categoricals)
6. Trains a single **global `GBTRegressor`** — one model for every SKU, no per-item models
7. Evaluates on the chronologically-held-out `test/` slice (RMSE, MAE, R²), plus zero-inflation diagnostics (fraction of zero rows, nonzero-only error) and feature importances
8. Writes `models/baseline_config.json` — the feature list, hyperparameters, importances, test metrics, and the known zero-inflation limitation, kept as a documentation/portfolio artifact (Phase 2 has no Spark, so this model never runs live there)

---
### Before you run
Uses the same Drive layout as notebook 1:
```
MyDrive/
└── SmartRetail/
    ├── data/
    │   └── processed/
    │       ├── train/
    │       └── test/
    └── models/
        └── baseline_config.json   ← written by this notebook
```


## Step 1 — Install PySpark

In [ ]:
!pip install pyspark==3.5.1 -q
print("PySpark installed.")

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## Step 3 — Configuration
**Only edit this cell.** All paths below derive from these two roots.

In [ ]:
# ── Edit these two lines to match your Drive layout ────────────────────────────
DATA_PATH   = "/content/drive/MyDrive/SmartRetail/data/processed/"
MODELS_PATH = "/content/drive/MyDrive/SmartRetail/models/"
# ───────────────────────────────────────────────────────────────────────────────

TRAIN_PARQUET       = DATA_PATH + "train/"
TEST_PARQUET        = DATA_PATH + "test/"
BASELINE_CONFIG_JSON = MODELS_PATH + "baseline_config.json"

TARGET_COL = "daily_transactions"

print("Config ready.")
print(f"  Train in    : {TRAIN_PARQUET}")
print(f"  Test in     : {TEST_PARQUET}")
print(f"  Config out  : {BASELINE_CONFIG_JSON}")

## Step 4 — Initialise SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SmartRetail-BaselineGBT")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "100")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Spark UI      : {spark.sparkContext.uiWebUrl}")

## Step 5 — Load Train / Test Parquet
Both slices come straight out of notebook 1 — `train/` is the earliest 80% of dates,
`test/` is the latest 20%. No row from `test/` was involved in producing `train/`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

train_df = spark.read.parquet(TRAIN_PARQUET)
test_df  = spark.read.parquet(TEST_PARQUET)

print(f"Train rows : {train_df.count():,}")
print(f"Test rows  : {test_df.count():,}")
train_df.printSchema()

## Step 6 — Add Lag Features
`daily_views` and `daily_addtocarts` describe what happened **during** the day — at
prediction time (start of day) we don't have them yet. We replace them with yesterday's
values per item, computed independently within each split so no information crosses the
train/test boundary.

In [ ]:
lag_window = Window.partitionBy("itemid").orderBy("event_date")

train_df = (
    train_df
    .withColumn("views_lag1",     F.lag("daily_views", 1).over(lag_window))
    .withColumn("addtocart_lag1", F.lag("daily_addtocarts", 1).over(lag_window))
)

test_df = (
    test_df
    .withColumn("views_lag1",     F.lag("daily_views", 1).over(lag_window))
    .withColumn("addtocart_lag1", F.lag("daily_addtocarts", 1).over(lag_window))
)

print("Lag features added.")
train_df.select("itemid", "event_date", "daily_views", "views_lag1",
                 "daily_addtocarts", "addtocart_lag1").orderBy("itemid", "event_date").show(8)

## Step 7 — Drop Leakage Columns
Now that yesterday's values are captured in the lag columns, the same-day columns must
be dropped — keeping them would let the model see the outcome it's supposed to predict.

In [ ]:
cols_to_drop = ["daily_views", "daily_addtocarts", "view_to_cart_ratio", "cart_to_purchase_ratio"]

train_df = train_df.drop(*cols_to_drop)
test_df  = test_df.drop(*cols_to_drop)

print(f"Dropped: {cols_to_drop}")
train_df.printSchema()

## Step 8 — Restrict to SKUs With Historical Sales
`daily_transactions` is mostly zeros — a model can reach deceptively high accuracy by
always predicting zero. We keep only SKUs that sold at least once during the **training**
window, and apply the same item population to `test/` so the evaluation metrics reflect
the same meaningful subset the model was actually trained for.

In [ ]:
items_with_sales = (
    train_df.filter(F.col(TARGET_COL) > 0)
    .select("itemid")
    .distinct()
)

n_before_train = train_df.select("itemid").distinct().count()
n_before_test  = test_df.select("itemid").distinct().count()

train_df = train_df.join(items_with_sales, on="itemid", how="inner")
test_df  = test_df.join(items_with_sales, on="itemid", how="inner")

print(f"SKUs with at least one training-window sale: {items_with_sales.count():,}")
print(f"Train SKUs : {n_before_train:,} → {train_df.select('itemid').distinct().count():,}")
print(f"Test SKUs  : {n_before_test:,} → {test_df.select('itemid').distinct().count():,}")

## Step 9 — Drop Null Lag Rows
The first date on record for each item has no "yesterday", so `views_lag1` /
`addtocart_lag1` are null there. These rows carry no usable signal for the lag features
and must be dropped before training.

In [ ]:
lag_cols = ["views_lag1", "addtocart_lag1"]

train_before = train_df.count()
test_before  = test_df.count()

train_df = train_df.dropna(subset=lag_cols)
test_df  = test_df.dropna(subset=lag_cols)

print(f"Train rows : {train_before:,} → {train_df.count():,}")
print(f"Test rows  : {test_before:,} → {test_df.count():,}")

## Step 10 — Encode Categorical Columns
`itemid` and `categoryid` are integers but the model must treat them as categories, not
continuous magnitudes. We fit `StringIndexer` on `train_df` only and reuse it on
`test_df` — `handleInvalid="keep"` routes any itemid/categoryid that only appears in the
test window to an extra "unseen" index instead of erroring.

`StringIndexer` attaches nominal (categorical) metadata to its output column. Left as-is,
`GBTRegressor` would treat `itemid_index` as a categorical feature and require
`maxBins >= itemid cardinality` — thousands of bins, impractical. We strip that metadata
so the tree model treats the index as an ordinary numeric feature instead.

**Gotcha:** `Column.alias(col, metadata={})` is a silent no-op — PySpark's `alias()`
checks `if metadata:` before applying it, and an empty dict is falsy in Python, so the
call falls through to the no-metadata branch and the *original* nominal metadata is
inherited unchanged. The override has to be a non-empty dict that explicitly declares
the column numeric.

In [ ]:
from pyspark.ml.feature import StringIndexer

itemid_indexer = StringIndexer(
    inputCol="itemid", outputCol="itemid_index", handleInvalid="keep"
).fit(train_df)

categoryid_indexer = StringIndexer(
    inputCol="categoryid", outputCol="categoryid_index", handleInvalid="keep"
).fit(train_df)

def strip_metadata(df, col_name):
    # metadata must be a non-empty dict (empty dict is falsy -> alias() silently
    # skips it and the original nominal ml_attr metadata leaks through unchanged)
    return df.withColumn(
        col_name, F.col(col_name).alias(col_name, metadata={"ml_attr": {"type": "numeric"}})
    )

train_indexed = itemid_indexer.transform(train_df)
train_indexed = categoryid_indexer.transform(train_indexed)
train_indexed = strip_metadata(train_indexed, "itemid_index")
train_indexed = strip_metadata(train_indexed, "categoryid_index")

test_indexed = itemid_indexer.transform(test_df)
test_indexed = categoryid_indexer.transform(test_indexed)
test_indexed = strip_metadata(test_indexed, "itemid_index")
test_indexed = strip_metadata(test_indexed, "categoryid_index")

print(f"itemid vocabulary size     : {len(itemid_indexer.labels):,}")
print(f"categoryid vocabulary size : {len(categoryid_indexer.labels):,}")
train_indexed.select("itemid", "itemid_index", "categoryid", "categoryid_index").show(5)

## Step 11 — Assemble Feature Vector
The final feature set going into the GBT, exactly as scoped for this baseline — no
per-item features beyond the encoded index, so the model generalises across SKUs
instead of memorising them.

In [ ]:
from pyspark.ml.feature import VectorAssembler

FEATURE_COLS = [
    "itemid_index", "categoryid_index", "base_price",
    "day_of_week", "day_of_month", "month", "week_of_year",
    "velocity_7d", "velocity_30d", "view_velocity_7d",
    "views_lag1", "addtocart_lag1",
]

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features")

train_ready = assembler.transform(train_indexed).select("features", TARGET_COL)
test_ready  = assembler.transform(test_indexed).select("features", TARGET_COL)

print(f"Feature vector size: {len(FEATURE_COLS)}")
train_ready.show(5, truncate=False)

## Step 12 — Train the Global GBTRegressor
One model, all SKUs — `itemid_index` and `categoryid_index` let it specialise per item
without maintaining a separate model per SKU.

In [ ]:
from pyspark.ml.regression import GBTRegressor

GBT_PARAMS = dict(
    featuresCol="features",
    labelCol=TARGET_COL,
    maxIter=100,
    maxDepth=6,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42,
)

gbt = GBTRegressor(**GBT_PARAMS)
gbt_model = gbt.fit(train_ready)

print("GBTRegressor trained.")
print(f"Number of trees : {gbt_model.getNumTrees}")
print(f"Total nodes     : {gbt_model.totalNumNodes}")

## Step 13 — Evaluate on the Held-Out Test Slice
Test predictions come entirely from the chronological future window — the model has
never seen these dates.

We also report the zero-transaction fraction of the test set and RMSE/MAE computed
only on rows with a real sale. A `log1p` target transform was tried and measured here
first — it left R²/RMSE essentially unchanged (0.3045→0.2998, 0.1787→0.1793) because
`log1p` corrects right-skew, not zero-inflation, and over mostly-{0,1} values it's
nearly linear. **96.65% of test rows have zero actual transactions**, and on the 1,516
rows that did sell, nonzero-only MAE was 0.72 against mostly-1 true values — the model
detects *which* items are likely to sell but not *how much*. That's the real ceiling on
this baseline; fixing it properly would mean a two-stage hurdle model (classify
"sells today?" then regress magnitude on the positive rows only), which is out of
scope for this baseline pass.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = gbt_model.transform(test_ready)

metrics = {}
for metric_name in ["rmse", "mae", "r2"]:
    evaluator = RegressionEvaluator(
        labelCol=TARGET_COL, predictionCol="prediction", metricName=metric_name
    )
    metrics[metric_name] = evaluator.evaluate(predictions)

print("── Test set metrics ───────────────────────────")
for name, value in metrics.items():
    print(f"  {name.upper():5s}: {value:.4f}")

# ── Diagnostics: how much of this is zero-inflation? ────────────────
zero_frac = predictions.filter(F.col(TARGET_COL) == 0).count() / predictions.count()
print(f"\nFraction of test rows with zero actual transactions: {zero_frac:.4f}")

nonzero_preds = predictions.filter(F.col(TARGET_COL) > 0)
print(f"Test rows with nonzero actual transactions: {nonzero_preds.count():,}")
nonzero_metrics = {}
for metric_name in ["rmse", "mae"]:
    evaluator = RegressionEvaluator(
        labelCol=TARGET_COL, predictionCol="prediction", metricName=metric_name
    )
    nonzero_metrics[metric_name] = evaluator.evaluate(nonzero_preds)
    print(f"  Nonzero-only {metric_name.upper():5s}: {nonzero_metrics[metric_name]:.4f}")

print("\n── Feature importances ─────────────────────────")
feature_importances = {
    col_name: float(round(importance, 4))
    for col_name, importance in zip(FEATURE_COLS, gbt_model.featureImportances.toArray())
}
for col_name, importance in sorted(feature_importances.items(), key=lambda kv: -kv[1]):
    print(f"  {col_name:20s}: {importance:.4f}")

## Step 14 — Write `baseline_config.json`
Phase 2 runs with no Spark, so the trained `GBTRegressor` object itself never executes
there — this config is a documentation/portfolio artifact capturing what was trained,
how it performed, and which features mattered.

In [ ]:
import os
import json

os.makedirs(MODELS_PATH, exist_ok=True)

baseline_config = {
    "model_type": "GBTRegressor",
    "target_column": TARGET_COL,
    "feature_columns": FEATURE_COLS,
    "hyperparameters": {k: v for k, v in GBT_PARAMS.items() if k not in ("featuresCol", "labelCol")},
    "test_metrics": {k: round(v, 4) for k, v in metrics.items()},
    "known_limitation": (
        "Target is zero-inflated: 96.65% of test rows have zero actual transactions. "
        "R2~0.30 reflects a model that predicts which items are likely to sell but not "
        "sale magnitude - nonzero-only metrics below show the real gap. A hurdle model "
        "(classify then regress) would be the correct fix but is out of scope for this baseline."
    ),
    "nonzero_test_metrics": {k: round(v, 4) for k, v in nonzero_metrics.items()},
    "zero_transaction_fraction": round(zero_frac, 4),
    "feature_importances": feature_importances,
    "itemid_vocabulary_size": len(itemid_indexer.labels),
    "categoryid_vocabulary_size": len(categoryid_indexer.labels),
    "train_rows": train_ready.count(),
    "test_rows": test_ready.count(),
}

with open(BASELINE_CONFIG_JSON, "w") as f:
    json.dump(baseline_config, f, indent=2)

print(f"Config written to: {BASELINE_CONFIG_JSON}")
print(json.dumps(baseline_config, indent=2))

## Step 15 — Final Summary

In [ ]:
print("═" * 55)
print("  NOTEBOOK 2 — BASELINE GBT TRAINING COMPLETE")
print("═" * 55)
print()
print(f"Test RMSE : {metrics['rmse']:.4f}")
print(f"Test MAE  : {metrics['mae']:.4f}")
print(f"Test R2   : {metrics['r2']:.4f}")
print()
print("Top 3 features by importance:")
for col_name, importance in sorted(feature_importances.items(), key=lambda kv: -kv[1])[:3]:
    print(f"  · {col_name}: {importance:.4f}")
print()
print(f"Config written to: {BASELINE_CONFIG_JSON}")
print()
print("What notebook 3 expects:")
print("  Read both TRAIN_PARQUET and TEST_PARQUET independently.")
print("  LSTM re-aggregates into 1-minute tumbling windows — does not reuse this notebook's features.")

spark.stop()
print()
print("SparkSession closed.")